# Formula 1 Prediction
## Chinese GP

In [3]:
import fastf1
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

from dotenv import load_dotenv
import os
import requests
load_dotenv()

API_KEY = os.getenv("API_KEY")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [4]:
fastf1.Cache.enable_cache("f1_cache")

session_aus_2026 = fastf1.get_session(2026, "Australia", "Q")
session_aus_2026.load()

session_chi_2026 = fastf1.get_session(2026, "China", "Q")
session_chi_2026.load()

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	No lap data for driver 18
core        WARNING 	No lap data for driver 55
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 18)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 55)
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
logger      WARNING 	Failed to load telemetry data!
req    

In [5]:
# Weather Data from Shanghai GP
lat, lon = 31.2304, 121.4737

url_from_var = f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
url_hardcoded = "http://api.openweathermap.org/data/2.5/forecast?lat=31.2304&lon=121.4737&appid=4d1aa06b0b99730fac4176f4cfa00256&units=metric"

print(repr(url_from_var))
print(repr(url_hardcoded))
print(url_from_var == url_hardcoded)

'http://api.openweathermap.org/data/2.5/forecast?lat=31.2304&lon=121.4737&appid=4d1aa06b0b99730fac4176f4cfa00256&units=metric'
'http://api.openweathermap.org/data/2.5/forecast?lat=31.2304&lon=121.4737&appid=4d1aa06b0b99730fac4176f4cfa00256&units=metric'
True


In [6]:
# Load the 2026
laps_2026 = session_chi_2026.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2026.dropna(inplace=True)

# convert lap and sector times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2026[f"{col} (s)"] = laps_2026[col].dt.total_seconds()

# aggregate sector times by driver
sector_times_2026 = laps_2026.groupby("Driver").agg({
    "Sector1Time (s)": "mean",
    "Sector2Time (s)": "mean",
    "Sector3Time (s)": "mean"
}).reset_index()

sector_times_2026["TotalSectorTime (s)"] = (
    sector_times_2026["Sector1Time (s)"] +
    sector_times_2026["Sector2Time (s)"] +
    sector_times_2026["Sector3Time (s)"]
)

print(sector_times_2026.sort_values("TotalSectorTime (s)").to_string())

china_quali = {
    "ANT": (24.003, 27.664, 40.387, 92.064, 1),
    "RUS": (24.012, 27.783, 40.491, 92.286, 2),
    "NOR": (27.158, 30.713, 47.535, 105.405, 3),
}

df = pd.DataFrame.from_dict(
    china_quali, orient="index",
    columns=["S1", "S2", "S3", "ChinaQuali_s", "ChinaGrid"]
)
df.index.name = "Driver"
df = df.reset_index()

df.head()

   Driver  Sector1Time (s)  Sector2Time (s)  Sector3Time (s)  TotalSectorTime (s)
14    NOR        27.158071        30.712786        47.534500           105.405357
12    LEC        26.361071        30.852143        48.652214           105.865429
21    VER        27.539571        31.181000        47.961786           106.682357
11    LAW        26.999364        31.012455        48.879636           106.891455
8     HAD        27.310357        31.362143        48.566714           107.239214
6     COL        29.429273        30.893273        47.050636           107.373182
2     ANT        27.299600        32.882400        47.660900           107.842900
5     BOT        27.026667        31.708667        49.217167           107.952500
17    PIA        27.515643        31.178500        49.492143           108.186286
16    PER        27.533750        32.515750        48.212000           108.261500
9     HAM        27.562923        30.837846        49.875538           108.276308
13    LIN       

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid
0,ANT,24.003,27.664,40.387,92.064,1
1,RUS,24.012,27.783,40.491,92.286,2
2,NOR,27.158,30.713,47.535,105.405,3


In [ ]:
df["UltimateLap_s"] = df["S1"] + df["S2"] + df["S3"]
pole = df["ChinaQuali_s"].min()
df["ChinaGapFromPole_s"] = (df["ChinaQuali_s"] - pole).round(3)

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222


In [ ]:
aus_grid = {
    "RUS": 1, "ANT": 2, "HAD": 3, "LEC": 4, "PIA": 5,
    "NOR": 6, "HAM": 7, "LAW": 8, "LIN": 9, "BOR": 10,
    "HUL": 11, "BEA": 12, "OCO": 13, "GAS": 14, "ALB": 15,
    "COL": 16, "ALO": 17, "PER": 18, "BOT": 19, "VER": 20,
    "SAI": 21, "STR": 22
}
df["AusGrid"] = df["Driver"].map(aus_grid)

aus_finish_pos = {
    "RUS": 1, "ANT": 2, "LEC": 3, "HAM": 4,
    "NOR": 5, "VER": 6, "BEA": 7, "LIN": 8,
    "BOR": 9, "GAS": 10, "OCO": 11, "ALB": 12,
    "LAW": 13, "COL": 14, "SAI": 15, "PER": 16,
    "STR": 17,
    "ALO": 18, "BOT": 19, "HAD": 20,
    "PIA": 21, "HUL": 22
}
df["AusFinishPos"] = df["Driver"].map(aus_finish_pos)

,Driver,S1,S2,S3,ChinaQuali_s,ChinaGrid,UltimateLap_s,ChinaGapFromPole_s,AusGrid,AusFinishPos
0,ANT,24.003,27.664,40.387,92.064,1,92.054,0.000,2,2
1,RUS,24.012,27.783,40.491,92.286,2,92.286,0.222,1,1


In [48]:
team_points = {
    "Mercedes": 55, "Ferrari": 40, "McLaren": 18, "Red Bull": 8,
    "Haas": 7, "Racing Bulls": 6, "Audi": 2, "Alpine": 1,
    "Williams": 1, "Cadillac": 1, "Aston Martin": 1,
}
max_pts = max(team_points.values())
team_score = {t: max(p, 0.5) / max_pts for t, p in team_points.items()}

driver_to_team = {
    "RUS": "Mercedes", "ANT": "Mercedes", "HAM": "Ferrari", "LEC": "Ferrari",
    "NOR": "McLaren", "PIA": "McLaren", "VER": "Red Bull", "HAD": "Red Bull",
    "BEA": "Haas", "OCO": "Haas", "LAW": "Racing Bulls", "LIN": "Racing Bulls",
    "HUL": "Audi", "BOR": "Audi", "GAS": "Alpine", "COL": "Alpine",
    "SAI": "Williams", "ALB": "Williams", "PER": "Cadillac", "BOT": "Cadillac",
}

In [ ]:
# load the 2024 Qatar session data
session_2024 = fastf1.get_session(2024, 24, "R")
session_2024.load()
laps_2024 = session_2024.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
laps_2024.dropna(inplace=True)

# convert lap and sector times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2024[f"{col} (s)"] = laps_2024[col].dt.total_seconds()

# aggregate sector times by driver
sector_times_2024 = laps_2024.groupby("Driver").agg({
    "Sector1Time (s)": "mean",
    "Sector2Time (s)": "mean",
    "Sector3Time (s)": "mean"
}).reset_index()

sector_times_2024["TotalSectorTime (s)"] = (
    sector_times_2024["Sector1Time (s)"] +
    sector_times_2024["Sector2Time (s)"] +
    sector_times_2024["Sector3Time (s)"]
)

# clean air race pace from racepace.py
clean_air_race_pace = {
    "VER": 93.191067, "HAM": 94.020622, "LEC": 93.418667, "NOR": 93.428600, "ALO": 94.784333,
    "PIA": 93.232111, "RUS": 93.833378, "SAI": 94.497444, "STR": 95.318250, "HUL": 95.345455,
    "OCO": 95.682128
}

# quali data from Abu Dhabi GP
qualifying_2025 = pd.DataFrame({
    "Driver": ["RUS", "VER", "PIA", "NOR", "HAM", "LEC", "ALO", "HUL", "ALB", "SAI", "STR", "OCO", "GAS"],
    "QualifyingTime (s)": [
        82.645,  # RUS
        82.207,  # VER
        82.437,  # PIA
        82.408,  # NOR
        83.394,  # HAM
        82.730,  # LEC
        82.902,  # ALO
        83.450,  # HUL
        83.416,  # ALB
        83.042,  # SAI
        83.097,  # STR
        82.913,  # OCO
        83.468   # GAS
    ]
})

lat, lon = 31.2304, 121.4737

qualifying_2025["CleanAirRacePace (s)"] = qualifying_2025["Driver"].map(clean_air_race_pace)
weather_url = f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
response = requests.get(weather_url)
weather_data = response.json()

In [ ]:
forecast_time = "2026-03-15 15:00:00"
forecast_data = next((f for f in weather_data["list"] if f["dt_txt"] == forecast_time), None)

rain_probability = forecast_data["pop"] if forecast_data else 0
temperature = forecast_data["main"]["temp"] if forecast_data else 20

# adjust qualifying time based on weather conditions
if rain_probability >= 0.75:
    qualifying_2025["QualifyingTime"] = qualifying_2025["QualifyingTime (s)"] * qualifying_2025["WetPerformanceFactor"]
else:
    qualifying_2025["QualifyingTime"] = qualifying_2025["QualifyingTime (s)"]

# add constructor"s data
team_points = {
    "McLaren": 800, "Mercedes": 459, "Red Bull": 426, "Williams": 137, "Ferrari": 382,
    "Haas": 73, "Aston Martin": 80, "Kick Sauber": 68, "Racing Bulls": 92, "Alpine": 22
}

max_points = max(team_points.values())
team_performance_score = {team: points / max_points for team, points in team_points.items()}

driver_to_team = {
    "VER": "Red Bull", "NOR": "McLaren", "PIA": "McLaren", "LEC": "Ferrari", "RUS": "Mercedes",
    "HAM": "Ferrari", "GAS": "Alpine", "ALO": "Aston Martin", "TSU": "Racing Bulls",
    "SAI": "Williams", "HUL": "Kick Sauber", "OCO": "Alpine", "STR": "Aston Martin"
}

qualifying_2025["Team"] = qualifying_2025["Driver"].map(driver_to_team)
qualifying_2025["TeamPerformanceScore"] = qualifying_2025["Team"].map(team_performance_score)

# merge qualifying and sector times data
merged_data = qualifying_2025.merge(sector_times_2024[["Driver", "TotalSectorTime (s)"]], on="Driver", how="left")
merged_data["RainProbability"] = rain_probability
merged_data["Temperature"] = temperature
merged_data["QualifyingTime"] = merged_data["QualifyingTime"]


valid_drivers = merged_data["Driver"].isin(laps_2024["Driver"].unique())
merged_data = merged_data[valid_drivers]

# define features (X) and target (y)
X = merged_data[[
    "QualifyingTime", "RainProbability", "Temperature", "TeamPerformanceScore", 
    "CleanAirRacePace (s)"
]]
y = laps_2024.groupby("Driver")["LapTime (s)"].mean().reindex(merged_data["Driver"])

# impute missing values for features
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

# train-test split
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.1, random_state=39)

# train XGBoost model
model = XGBRegressor(n_estimators=300, learning_rate=0.9, max_depth=3, random_state=39,  monotone_constraints="(1, 0, 0, -1, -1)")
model.fit(X_train, y_train)
merged_data["PredictedRaceTime (s)"] = model.predict(X_imputed)

# sort the results to find the predicted winner
final_results = merged_data.sort_values(by=["PredictedRaceTime (s)", "QualifyingTime"]).reset_index(drop=True)
print(final_results[["Driver", "PredictedRaceTime (s)"]])

# sort results and get top 3
podium = final_results.loc[:7, ["Driver", "PredictedRaceTime (s)"]]
print("\n🏆 Predicted in the Top 3 🏆")
print(f"🥇 P1: {podium.iloc[0]["Driver"]}")
print(f"🥈 P2: {podium.iloc[1]["Driver"]}")
print(f"🥉 P3: {podium.iloc[2]["Driver"]}")
y_pred = model.predict(X_test)
print(f"Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")

# Plot feature importances
feature_importance = model.feature_importances_
features = X.columns 

plt.figure(figsize=(8,5))
plt.barh(features, feature_importance, color="skyblue")
plt.xlabel("Importance")
plt.title("Feature Importance in Race Time Prediction")
plt.tight_layout()
plt.show()